# Separação e threshold para AVC

Este notebook usa o pipeline selecionado pelo treino e percorre thresholds para a classe `stroke=1`. A busca usa probabilidades out-of-fold. O treino compara SMOTENC e SMOTENC + RandomUnderSampler com XGBoost ponderado sem reamostragem; a seleção principal considera AUPRC, métrica de ranking adequada à classe rara e independente do threshold.

Os thresholds que maximizam precision e F1 são escolhidos somente no OOF e avaliados uma vez no holdout. As tabelas incluem métricas das classes 0 e 1, accuracy geral, recall, alertas e matriz de confusão.

Precision e F1 ≥ 0,90 são metas, não garantias. Com poucos casos de AVC, esses valores podem ser inviáveis ou instáveis. O notebook não altera o threshold nem o artefato do modelo.

In [2]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from config.paths_config import DATA_PATH, MODELS_DIR
from config.training_config import CV_FOLDS, RANDOM_STATE, TEST_SIZE
from src.data.loader import load_dataset
from src.features.engineering import split_features_target

bundle = joblib.load(MODELS_DIR / 'stroke_pipeline.joblib')
X, y = split_features_target(load_dataset(DATA_PATH))
X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
pipeline = clone(bundle['pipeline'])  # usa o sampler vencedor registrado no artefato
print(f"Modelo: {bundle['model_name']} | sampler: {bundle['sampler']} | AVC no treino: {int(y_train.sum())} | AVC no holdout: {int(y_holdout.sum())}")

ModuleNotFoundError: No module named 'config'

## Busca OOF do threshold para AVC=1

O `for` calcula precision, F1, recall e accuracy em cada threshold candidato. Imputação, encoding e balanceamento (quando aplicável) são ajustados dentro de cada fold; o holdout permanece intocado durante a seleção.

In [ ]:
oof_probability = cross_val_predict(
    clone(pipeline), X_train, y_train, cv=cv,
    method='predict_proba', n_jobs=1
)[:, 1]
thresholds = np.unique(np.r_[0.0, oof_probability, 1.0])
rows = []
for threshold in thresholds:
    prediction = (oof_probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, prediction, labels=[0, 1]).ravel()
    rows.append({
        'threshold': float(threshold),
        'accuracy': accuracy_score(y_train, prediction),
        'precision_classe_1': precision_score(y_train, prediction, zero_division=0),
        'f1_classe_1': f1_score(y_train, prediction, zero_division=0),
        'recall_classe_1': recall_score(y_train, prediction, zero_division=0),
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'alertas_AVC': int(prediction.sum())
    })
threshold_table = pd.DataFrame(rows)
threshold_table['precision_e_f1_ge_0_90'] = (
    (threshold_table['precision_classe_1'] >= 0.90) &
    (threshold_table['f1_classe_1'] >= 0.90)
)
feasible = threshold_table[threshold_table['precision_e_f1_ge_0_90']]
best_precision = threshold_table.loc[threshold_table['precision_classe_1'].idxmax()]
best_f1 = threshold_table.loc[threshold_table['f1_classe_1'].idxmax()]
if not feasible.empty:
    feasible = feasible.assign(joint_mean=feasible[['precision_classe_1', 'f1_classe_1']].mean(axis=1))
    best_joint = feasible.loc[feasible['joint_mean'].idxmax()]
    joint_criterion = 'precision e F1 >= 0.90 (melhor média)'
else:
    best_joint = best_f1
    joint_criterion = 'sem threshold viável; fallback: melhor F1'

selected_rows = [
    {'critério': 'maior precision classe 1', **best_precision.to_dict()},
    {'critério': 'maior F1 classe 1', **best_f1.to_dict()},
    {'critério': joint_criterion, **best_joint.to_dict()},
]
candidates = pd.DataFrame(selected_rows)
print(f"Thresholds que atingem precision e F1 >= 0,90 no OOF: {len(feasible)}")
if feasible.empty:
    print('Meta simultânea não atingida no OOF; exibindo os melhores resultados possíveis por métrica.')
else:
    display(feasible.sort_values('joint_mean', ascending=False).head(10).round(4).to_string(index=False))
display(candidates[['critério', 'threshold', 'accuracy', 'precision_classe_1',
                    'f1_classe_1', 'recall_classe_1', 'alertas_AVC', 'FP', 'FN']]
        .round(4).to_string(index=False))

## Curvas das métricas OOF por threshold

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(threshold_table['threshold'], threshold_table['precision_classe_1'], label='Precision AVC=1')
ax.plot(threshold_table['threshold'], threshold_table['f1_classe_1'], label='F1 AVC=1')
ax.plot(threshold_table['threshold'], threshold_table['recall_classe_1'], label='Recall AVC=1')
ax.plot(threshold_table['threshold'], threshold_table['accuracy'], label='Accuracy geral', alpha=0.75)
ax.axhline(0.90, color='grey', linestyle='--', label='Meta 0,90')
for _, row in candidates.iterrows():
    ax.axvline(row['threshold'], linestyle=':', alpha=0.5)
ax.set(xlabel='Threshold', ylabel='Métrica', ylim=(0, 1.02), title=f"{bundle['model_name']} + {bundle['sampler']}: OOF")
ax.legend(ncol=2); ax.grid(alpha=0.2); fig.tight_layout()

## Holdout: métricas por classe nos thresholds candidatos

A tabela abaixo avalia os thresholds selecionados no OOF no conjunto holdout. Para cada classe mostra precision, F1, recall e acurácia one-vs-rest; a accuracy geral e a matriz de confusão são incluídas. Um threshold com alta precision pode sinalizar quase ninguém, então verifique também recall, TP e FN.

In [ ]:
holdout_pipeline = clone(pipeline).fit(X_train, y_train)
holdout_probability = holdout_pipeline.predict_proba(X_holdout)[:, 1]
evaluation_rows = []
for _, row in candidates.iterrows():
    prediction = (holdout_probability >= row['threshold']).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_holdout, prediction, labels=[0, 1]).ravel()
    overall_accuracy = accuracy_score(y_holdout, prediction)
    for class_label in (0, 1):
        binary_true = (y_holdout.to_numpy() == class_label).astype(int)
        binary_pred = (prediction == class_label).astype(int)
        class_tn, class_fp, class_fn, class_tp = confusion_matrix(
            binary_true, binary_pred, labels=[0, 1]
        ).ravel()
        evaluation_rows.append({
            'critério OOF': row['critério'], 'threshold': row['threshold'],
            'classe': class_label, 'accuracy geral': overall_accuracy,
            'accuracy classe OVR': (class_tp + class_tn) / len(y_holdout),
            'precision': precision_score(binary_true, binary_pred, zero_division=0),
            'F1': f1_score(binary_true, binary_pred, zero_division=0),
            'recall': recall_score(binary_true, binary_pred, zero_division=0),
            'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp, 'alertas AVC': int(prediction.sum())
        })
metrics_by_class = pd.DataFrame(evaluation_rows)
display(metrics_by_class.round(4).to_string(index=False))